# **Trainer**

## **1. Dataset initialization**

As in the previous notebook. First set your dataset directory, initialize it and create the dataloaders for training.

In [ ]:
# Make repo root importable for this session (zero packaging)
import sys, pathlib
repo_root = pathlib.Path.cwd().parent
sys.path.insert(0, str(repo_root))
%load_ext autoreload
%autoreload 2

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

# 1. Imports from your repo
from SCAPES.data.dataset import AtomSequenceDataset
from SCAPES.auxiliar.encodec_wrapper import EncodecProcessor
from SCAPES.training.FlowModel_trainer import FlowTrainer, get_model_configs
from SCAPES.models.factorization import LocalEncoder
from SCAPES.models.flow import FlowModel

# Fill this with the path to your dataset
path_to_dataset = "path/to/your/dataset"  # Change this to your dataset path <---------------------------

# Default configurations
segment_length, context_length, hop_size = 5, 5, 1

# dataset initialization (CPU is mandatory here, trainer moves batches to GPU)
dataset = AtomSequenceDataset(
    dataset_path=path_to_dataset, 
    segment_length=segment_length,
    context_length=context_length,
    hop_size=hop_size,
    requested_keys=[
        "latent_past", "latent_present", 
        "scale_past", "scale_present", 
        "clap_context_win", "index"
    ],
    device="cpu", # Leave dataset on CPU, trainer moves batches to GPU
    verbose=True
)

# create dataloaders for training and validation
train_split, val_split = dataset.get_splits()
train_loader = DataLoader(train_split, batch_size=16, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_split,   batch_size=16, shuffle=False)

## **2. Model Configuration**

Pick a model size between `small`, `medium` and `large`. If unsure just leave it as it is :)

In [ ]:
CONFIG_TYPE = "medium" # Change the size of yoru model <---------------------------

# Get the configurations to initialize the models based on the dataset and CONFIG_TYPE
LocalEncoder_config, FlowModel_config = get_model_configs(
    size             = CONFIG_TYPE,
    segment_length   = segment_length,
    frames_per_atom  = dataset.atoms_frames,         # Dynamically pulled
    atoms_hop_frames = dataset.atoms_hop_frames,    # Dynamically pulled
    crossfade_frames = dataset.crossfade_frames,    # Dynamically pulled
    LocalEncoder_time_entanglement = True,
    LocalEncoder_temporal_compression = 1
)

# Set your device
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load the audio processor
processor_48k = EncodecProcessor(sr=48000, streamable=True, device=device)

# Initialize models using the dictionaries (**kwargs unpacking)
local_encoder = LocalEncoder(**LocalEncoder_config).to(device)
flow_model    = FlowModel(**FlowModel_config, device=device).to(device)

# count the parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"LocalEncoder parameters: {count_parameters(local_encoder):,}")
print(f"FlowModel parameters: {count_parameters(flow_model):,}")

# Initialize optimizer for both models
optimizer = AdamW(list(flow_model.parameters()) + list(local_encoder.parameters()), lr=1e-4)

## **3. Train!**

Although optional, you can listen to some audio reconstruction during training. In order to do so, fill the `TARGET_FILES` list with names of your files. Make sure to specify a model_path so you save your trained model.

In [ ]:
# PASS A LIST OF FILES YOU WANT TO MONITOR
TARGET_FILES = [] # Example files from your val split <---------------------------

# model path
MODEL_SAVE_PATH = "path/to/your/model" # Make sure this is unique for your new model! <---------------------------

# Initialize the trainer
trainer = FlowTrainer(
    model           = flow_model,
    local_encoder   = local_encoder,
    train_loader    = train_loader,
    val_loader      = None,
    dataset         = dataset,
    processor       = processor_48k,
    optimizer       = optimizer,
    model_config    = FlowModel_config, 
    encoder_config  = LocalEncoder_config,  
    model_path      = MODEL_SAVE_PATH,
    context_source  = "clap",
    val_audio_files = TARGET_FILES,
    device          = device,
    past_dropout    = 0.25
)

# Start Training
trainer.train(epochs=50, audio_val_freq=10, val_nfe=16)